This is an update to my [2012 post on raking weights in R](/blog/2012/raking/). Same idea, same Chilean CEP survey example — but this time in Python with [weightpipe](https://github.com/sdaza/weightpipe), a package I wrote for declarative survey weighting recipes.

The conceptual background (what raking is, when to use it, truncation and design-effect checks) is unchanged; see the original post for that discussion. Here I focus on the Python workflow: define population margins, build a `Recipe`, calibrate, trim, and compare estimates.

Install from GitHub (not on PyPI yet):

```bash
pip install "git+https://github.com/sdaza/weightpipe.git"
# or
uv add "git+https://github.com/sdaza/weightpipe.git"
```


## Data

Again I use the Opinion Public Survey CEP, July–August 2012, to estimate presidential approval
([data](https://raw.githubusercontent.com/sdaza/sdaza.github.io/main/_R/data/cep.csv)).
Five variables enter the raking: `sex`, `agecat`, `ses`, `region`, and `area`.


In [1]:
import pandas as pd

pd.set_option("display.notebook_repr_html", False)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", None)

from weightpipe import (
    Recipe,
    boot_proportion,
    bootstrap_weights,
    collect_weights,
    design_effect,
    estimate,
)

dat = pd.read_csv(
    "https://raw.githubusercontent.com/sdaza/sdaza.github.io/main/_R/data/cep.csv"
)
dat["base"] = 1.0

for code, name in [(1, "approve"), (2, "disapprove"), (3, "unsure"), (9, "dk")]:
    dat[name] = (dat["approval"] == code).astype(int)

dat[["sex", "agecat", "ses", "region", "area", "pond", "approval"]].head()


   sex  agecat  ses  region  area  pond  approval
0    1       2    2      13     1 1.977         2
1    1       5    2      13     1 1.243         1
2    2       2    3       9     2 0.514         2
3    1       5    3       9     1 0.421         1
4    1       5    4      10     1 0.526         1

Sample margins (unweighted) for the raking variables:


In [2]:
for var in ["sex", "agecat", "ses", "region", "area"]:
    print(f"\n{var}")
    print(
        dat[var]
        .value_counts(normalize=True)
        .sort_index()
        .rename_axis(None)
        .round(3)
        .to_string()
    )



sex
1   0.407
2   0.593

agecat
1   0.124
2   0.159
3   0.177
4   0.194
5   0.346

ses
1   0.039
2   0.108
3   0.365
4   0.448
5   0.040

region
1    0.013
2    0.042
3    0.014
4    0.042
5    0.099
6    0.055
7    0.062
8    0.131
9    0.063
10   0.049
11   0.004
12   0.011
13   0.376
14   0.026
15   0.013

area
1   0.837
2   0.163


## Population targets

Population shares come from the Chilean Census 2002 (`sex`, `agecat`, `region`, `area`) and the Bicentenario Survey 2009 (`ses`) — same targets as in 2012. Rounded census vectors may not sum exactly to one; `weightpipe` renormalizes them by default (`force1=True`), as in `anesrake`.


In [3]:
# Chilean Census 2002
sex = {1: 0.49, 2: 0.51}  # 1 male, 2 female
agecat = {
    1: 0.163,  # 18-24
    2: 0.203,  # 25-34
    3: 0.195,  # 35-44
    4: 0.187,  # 45-54
    5: 0.253,  # 55+
}
region = {
    1: 0.015, 2: 0.031, 3: 0.016, 4: 0.039, 5: 0.102,
    6: 0.051, 7: 0.059, 8: 0.123, 9: 0.056, 10: 0.046,
    11: 0.006, 12: 0.010, 13: 0.408, 14: 0.023, 15: 0.013,
}
area = {1: 0.869, 2: 0.131}  # 1 urban, 2 rural

# Bicentenario Survey 2009
ses = {1: 0.109, 2: 0.184, 3: 0.261, 4: 0.364, 5: 0.083}  # abc1..e

proportions = {
    "sex": sex,
    "agecat": agecat,
    "ses": ses,
    "region": region,
    "area": area,
}


## Raking with `weightpipe`

A `Recipe` starts from base weights (here uniform `base = 1`), then chains steps. For this update I use:

- `step_calibrate(method="raking", proportions=...)` — IPF to the population margins
- `step_trim(max_ratio=5, reference="value")` — truncate weights above 5 (same spirit as `anesrake`'s `cap = 5`), redistributing excess so the total stays put

Unlike `anesrake`, variable selection (`pctlim`, `nlim`) is left to you: pass the margins you want to calibrate to.


In [4]:
recipe = (
    Recipe(dat, base_weight="base")
    .step_calibrate(
        method="raking",
        proportions=proportions,
        max_iter=100,
        tol=1e-8,
    )
    .step_trim(max_ratio=5.0, reference="value", redistribute=True)
)

fitted = recipe.prep(warn=False)
weighted = collect_weights(fitted, keep_intermediate=True)

print("n =", len(weighted))
print("sum(weight) =", round(float(weighted["weight"].sum()), 3))
print("min / max weight =", round(weighted["weight"].min(), 3), "/", round(weighted["weight"].max(), 3))
print("Kish deff =", round(design_effect(fitted), 3))
print("converged =", fitted.diagnostics["steps"]["calibrate"]["converged"])
print("iterations =", fitted.diagnostics["steps"]["calibrate"]["iterations"])
weighted[["weight"]].describe().T


n = 1512
sum(weight) = 1512.0
min / max weight = 0.332 / 4.304
Kish deff = 1.38
converged = True
iterations = 12


          count  mean   std   min   25%   50%   75%   max
weight 1512.000 1.000 0.616 0.332 0.618 0.804 1.105 4.304

Kish's approximate design effect from unequal weighting is again about **1.38** — the same figure as in the original R post. Weighting loss is $$L_w = \mathrm{deff} - 1 \approx 0.38$$, under the usual caveats (no clustering in this approximation).


### Approval estimates with bootstrap CIs

Approval codes: 1 = approve, 2 = disapprove, 3 = unsure, 9 = don't know.
Use `estimate` on binary indicators instead of a custom weighted tabulation.
With no strata/PSU in the CEP file, this is an unequal-weight bootstrap that
re-runs the full raking recipe in each replicate.


In [5]:
estimate(
    recipe,
    "approve",
    estimand="proportion",
    fitted=fitted,
    variance="bootstrap",
    replicates=400,
    seed=42,
).round(3)


   estimate    se  ci_lower  ci_upper  level  R_used    estimand variable   variance
0     0.298 0.013     0.272     0.325  0.950     400  proportion  approve  bootstrap

In [6]:
boot = bootstrap_weights(
    recipe,
    replicates=400,
    seed=42,
    point=fitted,
)

approval_ci = pd.concat(
    [
        boot_proportion(boot, name).assign(category=name)
        for name in ["approve", "disapprove", "unsure", "dk"]
    ],
    ignore_index=True,
)[["category", "estimate", "se", "ci_lower", "ci_upper"]]

approval_ci.round(3)


     category  estimate    se  ci_lower  ci_upper
0     approve     0.298 0.013     0.272     0.325
1  disapprove     0.521 0.013     0.494     0.547
2      unsure     0.161 0.010     0.141     0.181
3          dk     0.020 0.003     0.013     0.027

## Raking on top of existing survey weights

The CEP file includes `pond` weights (max ≈ 17.6). As before, documentation of how they were built is thin. We can still use them as the base weight and rake (here only on `ses` and `region`, the margins that were most off after applying `pond` in the 2012 analysis).


In [7]:
print("pond summary")
print(dat["pond"].describe().round(3).to_string())
print("Kish deff (pond) =", round(design_effect(dat["pond"]), 3))

recipe_pond = (
    Recipe(dat, base_weight="pond")
    .step_calibrate(
        method="raking",
        proportions={"ses": proportions["ses"], "region": proportions["region"]},
        max_iter=100,
        tol=1e-8,
    )
    .step_trim(max_ratio=5.0, reference="value", redistribute=True)
)

fitted_pond = recipe_pond.prep(warn=False)
weighted_pond = collect_weights(fitted_pond)

print("Kish deff (raked pond) =", round(design_effect(fitted_pond), 3))
print("min / max weight =", round(weighted_pond["weight"].min(), 3), "/", round(weighted_pond["weight"].max(), 3))


pond summary
count   1512.000
mean       1.000
std        1.044
min        0.015
25%        0.455
50%        0.786
75%        1.235
max       17.563
Kish deff (pond) = 2.088
Kish deff (raked pond) = 1.81
min / max weight = 0.055 / 5.0


In [8]:
boot_pond = bootstrap_weights(
    recipe_pond,
    replicates=400,
    seed=42,
    point=fitted_pond,
)

approval_pond_ci = pd.concat(
    [
        boot_proportion(boot_pond, name).assign(category=name)
        for name in ["approve", "disapprove", "unsure", "dk"]
    ],
    ignore_index=True,
)[["category", "estimate", "se", "ci_lower", "ci_upper"]]

approval_pond_ci.round(3)


     category  estimate    se  ci_lower  ci_upper
0     approve     0.292 0.013     0.266     0.319
1  disapprove     0.531 0.014     0.504     0.557
2      unsure     0.154 0.013     0.128     0.179
3          dk     0.023 0.004     0.015     0.031

The story matches the R version: raking from uniform bases and raking from `pond`
give similar approval estimates, and the bootstrap intervals overlap. The larger
gap remains between the original `pond` weights and the full demographic rake.

## Recipe shape (for reuse)

```python
from weightpipe import Recipe, collect_weights, design_effect, estimate

recipe = (
    Recipe(dat, base_weight="base")  # or "pond"
    .step_calibrate(method="raking", proportions=proportions)
    .step_trim(max_ratio=5.0, reference="value", redistribute=True)
)
fitted = recipe.prep()
weights = collect_weights(fitted)
design_effect(fitted)

estimate(
    recipe,
    "approve",
    estimand="proportion",
    fitted=fitted,
    variance="bootstrap",
    replicates=400,
    seed=42,
)
```

For clustered designs, pass a `Design(...)` into `Recipe.from_design` and let
`estimate` use bootstrap or jackknife variance. See the
[weightpipe README](https://github.com/sdaza/weightpipe).
